In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipynb.fs.full.preprocessing import process_the_data, read_the_data

<h3> Looking at the steps calculated by DeltabotController

In [ ]:
mesh0045_27Oct = pd.read_csv("processed_data/27Oct_0045_mesh.csv")

<h1>snaking x and y data

In [ ]:
#gave those x and y values to the 16Oct MLP 
#output
pred_snake_steps = pd.read_csv("predictions/16Oct_MLP_Window_Pred_snake_mesh.csv", names=["s0-1","s0", "s0+1", "s1-1", "s1", "s1+1", "s2-1", "s2", "s2+1"], dtype="float32")
display(pred_snake_steps.head())
print("Predicted motor values Max/Min")
print(max(pred_snake_steps["s0"]))
print(max(pred_snake_steps["s1"]))
print(max(pred_snake_steps["s2"]))
print(min(pred_snake_steps["s0"]))
print(min(pred_snake_steps["s1"]))
print(min(pred_snake_steps["s2"]))

pred_snake_steps.head()

In [ ]:
pred_snake_steps['sum_motors'] = pred_snake_steps["s0"] + pred_snake_steps["s1"] + pred_snake_steps["s2"]
# display(pred_snake_steps['sum_motors'])
print(max(pred_snake_steps["sum_motors"]))
print(min(pred_snake_steps["sum_motors"]))

print(pred_snake_steps.shape[0])

#adjust motor values so they always sum to 0

pred_snake_steps["s0_adj"] = pred_snake_steps["s0"] - (pred_snake_steps["sum_motors"]/3)
pred_snake_steps["s1_adj"] = pred_snake_steps["s1"] - (pred_snake_steps["sum_motors"]/3)
pred_snake_steps["s2_adj"] = pred_snake_steps["s2"] - (pred_snake_steps["sum_motors"]/3)

pred_snake_steps['sum_motors_adj'] = pred_snake_steps["s0_adj"] + pred_snake_steps["s1_adj"] + pred_snake_steps["s2_adj"]
# display(pred_snake_steps['sum_motors'])
print(max(pred_snake_steps["sum_motors_adj"]))
print(min(pred_snake_steps["sum_motors_adj"]))

pred_snake_steps.head()


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15,5), constrained_layout=True)
for i, motor in enumerate(["s0", "s1", "s2"]):
    im = axs[i].scatter(pred_snake_steps[motor].index, pred_snake_steps[motor], c=pred_snake_steps.index)
    axs[i].set_xlabel("index")
    axs[i].set_ylabel("MLP Predicted Steps Snake")
    fig.colorbar(im, ax=axs[i]) 
fig.suptitle("Predictions from Standard MLP Snake Scan")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15,5), constrained_layout=True)
for i, motor in enumerate(["s0_adj", "s1_adj", "s2_adj"]):
    im = axs[i].scatter(pred_snake_steps[motor].index, pred_snake_steps[motor], c=pred_snake_steps.index)
    axs[i].set_xlabel("index")
    axs[i].set_ylabel("MLP Predicted Steps Snake")
    fig.colorbar(im, ax=axs[i])
fig.suptitle("Predictions from Standard MLP Snake Scan - motor values adjusted to sum to 0")

<h2> Comparing the 2 datasets

In [ ]:
mesh0045_27Oct_dropped = mesh0045_27Oct.drop([0,624])

fig, axs = plt.subplots(1, 3, figsize=(15,5), constrained_layout=True)
for i, motor in enumerate(["s0", "s1", "s2"]):
    im = axs[i].scatter(mesh0045_27Oct_dropped[motor], pred_snake_steps[motor], c=mesh0045_27Oct_dropped["point"])
    axs[i].set_xlabel("Calculated Steps")
    axs[i].set_ylabel("MLP Predicted Steps")
    fig.colorbar(im, ax=axs[i])  
    axs[i].set_title(motor)
    lims = [-8000, 8000]
    # axs[i].set_xlim(lims)
    # axs[i].set_ylim(lims)
    _ = axs[i].plot(lims, lims, color="red")

fig.suptitle("DeltabotController Motor values vs Standard MLP Predictions for snake scan")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15,5), constrained_layout=True)
for i, motor in enumerate(["s0", "s1", "s2"]):
    im = axs[i].scatter(mesh0045_27Oct_dropped[motor], pred_snake_steps[f"{motor}_adj"], c=mesh0045_27Oct_dropped["point"])
    axs[i].set_xlabel("Calculated Steps")
    axs[i].set_ylabel("MLP Predicted Steps")
    fig.colorbar(im, ax=axs[i])  
    axs[i].set_title(motor)
    lims = [-8000, 8000]
    # axs[i].set_xlim(lims)
    # axs[i].set_ylim(lims)
    _ = axs[i].plot(lims, lims, color="red")

fig.suptitle("DeltabotController Motor values vs Standard MLP Predictions for snake scan - values adjusted to sum to 0")

In [ ]:
pred_snake_steps[["s0_adj", "s1_adj", "s2_adj"]].astype("int32").to_csv("predictions/adjusted/16Oct_MLP_Window_snake.csv", index=False)

In [ ]:
data = pd.read_csv("predictions/adjusted/16Oct_MLP_Window_snake.csv")
points = list(data.itertuples(name=None, index=False))
print(points)

In [ ]:
# list(pred_snake_steps.itertuples(index=False, name=None))

<h1>Output from Scan using predicted motor values

In [ ]:
path = "scripts/scans/MLP"
scan = "0006_MLP_16Oct"

x_conversion = 0.1/11
y_conversion = x_conversion

start_cx = 314 #px #scan 94
start_cy = 264 #px #scan 94

scan_MLP = process_the_data(scan, path, x_conversion, y_conversion, start_cx, start_cy)
scan_MLP.name = "scan_MLP"

In [ ]:
plt.scatter(scan_MLP["cx"], scan_MLP["cy"])

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15,5), constrained_layout=True)

im = axs[0].scatter(mesh0045_27Oct["x"], mesh0045_27Oct["y"], c=mesh0045_27Oct["point"])
axs[0].set_xlabel("x")
axs[0].set_ylabel("y")
axs[0].set_title("Aim x and y")

i = 1
for scan in mesh0045_27Oct, scan_MLP:
    subplot = axs[i]
    subplot.scatter(scan["cx_mm"], scan["cy_mm"], c=scan["point"])
    subplot.set_xlabel("cx_mm")
    subplot.set_ylabel("cy_mm")
    # subplot.set_xlim(-1.2, 1.2)
    # subplot.set_ylim(-1.2,1.2)

    # fig.colorbar(im, ax=axs[i])  
    # subplot.set_title(scan.name)
    i = i+1

fig.colorbar(im, ax=axs[2])  